# Seminar 2. Vectorization

## TF-IDF

Term frequency, $\text{tf}(t,d)$, is the relative frequency of term $t$ within document $d$,

$$
\text{tf}(t,d)=\frac{f_{t,d}}{\sum_{t' \in d}f_{t',d}},
$$

where $f_{t,d}$ is the raw count of a term in a document, i.e., the number of times that term $t$ occurs in document $d$. Note the denominator is simply the total number of terms in document $d$ (counting each occurrence of the same term separately).

The inverse document frequency is a measure of how much information the word provides, i.e., how common or rare it is across all documents. It is the logarithmically scaled inverse fraction of the documents that contain the word (obtained by dividing the total number of documents by the number of documents containing the term, and then taking the logarithm of that quotient):

$$
\text{idf}(t,D)=\log{\frac{N}{n_t}}
$$

with
- $D$: is the set of all documents in the corpus
- $N=|D|$: total number of documents in the corpus
- $n_t=|\{d \in D:t \in d\}|$: number of documents where the term $t$ appears (i.e., $\text{tf}(t,d) \neq 0$. If the term is not in the corpus, this will lead to a division-by-zero. It is therefore common to adjust the numerator to $1+N$ and the denominator to $1+|\{d \in D:t \in d\}|$.

Then tf–idf is calculated as

$$
\text{tfidf}(t,d,D)=\text{tf}(t,d)*\text{idf}(t,D)
$$

### Low-level implementation

In [1]:
import math
from collections import Counter
from typing import Dict, List

documents = [
    "cat dog bird",
    "cat rat cheese rat",
    "dog cat friend human"
]


Step 1. Tokenization

In [2]:
tokenized_docs = [doc.split() for doc in documents]
print("Tokenized docs:", tokenized_docs)


Tokenized docs: [['cat', 'dog', 'bird'], ['cat', 'rat', 'cheese', 'rat'], ['dog', 'cat', 'friend', 'human']]


Step 2. Count TF

Task 1. Given the formula, write function for counting TF.

$\text{tf}(t,d)=\frac{f_{t,d}}{\sum_{t' \in d}f_{t',d}},$

where $f_{t,d}$ is the raw count of a term in a document, i.e., the number of times that term $t$ occurs in document $d$. Note the denominator is simply the total number of terms in document $d$ (counting each occurrence of the same term separately).

In [3]:
def compute_tf(tokens: List[str]) -> Dict[str,float]:
    """Count TF for tokens in one document."""
    freq = {}

    for token in tokens:
        if token not in freq.keys():
            freq[token] = 1
        else:
            freq[token] += 1

    tf = {}

    for key, value in freq.items():
        tf[key] = value/len(tokens)

    return tf



In [4]:
tf_doc0 = compute_tf(tokenized_docs[0])
print("\nTF for document:", tf_doc0)



TF for document: {'cat': 0.3333333333333333, 'dog': 0.3333333333333333, 'bird': 0.3333333333333333}


Step 3. Count IDF

Task 2. Given the formula, write function for counting IDF.

$\text{idf}(t,D)=\log{\frac{N}{n_t}}$

with
- $D$: is the set of all documents in the corpus
- $N=|D|$: total number of documents in the corpus
- $n_t=|\{d \in D:t \in d\}|$: number of documents where the term $t$ appears (i.e., $\text{tf}(t,d) \neq 0$. If the term is not in the corpus, this will lead to a division-by-zero. It is therefore common to adjust the numerator to $1+N$ and the denominator to $1+|\{d \in D:t \in d\}|$.

In [5]:
def compute_idf(tokenized_docs:List[List[str]]) -> Dict[str,float]:
    """Count IDF for all tokens in corpus."""
    num_docs = len(tokenized_docs)
    # Step 1. For each term count the number of documents in which it occurs
    freq = {}
    for doc in tokenized_docs:
        unique_terms = set(doc)
        for term in unique_terms:
            freq[term] = freq.get(term, 0) + 1
    # Step 2. Count IDF
    idf = {term: math.log((num_docs + 1) / (1 + df)) for term, df in freq.items()}
    return idf


In [6]:
idf_dict = compute_idf(tokenized_docs)
print("\nIDF for all tokens:", idf_dict)



IDF for all tokens: {'bird': 0.6931471805599453, 'cat': 0.0, 'dog': 0.28768207245178085, 'rat': 0.6931471805599453, 'cheese': 0.6931471805599453, 'friend': 0.6931471805599453, 'human': 0.6931471805599453}


Step 4. Count TF-IDF

In [7]:
tfidf_doc0 = {term: tf_doc0.get(term, 0) * idf_dict.get(term, 0) for term in tf_doc0.keys()}
print("\nTF-IDF for document 0:", tfidf_doc0)



TF-IDF for document 0: {'cat': 0.0, 'dog': 0.09589402415059362, 'bird': 0.23104906018664842}


Step 5. Vectors

Task 3.

Create a matrix of size $n \times m$ where $n$ is the number of documents and $m$ is the number of unique terms.

In [8]:
import pandas as pd


In [9]:
# @title Try to write your own code in the cell above or check the implementation in this cell

# create corpus vocabulary (in this example I decided to preserve the order of terms' appearance.
# If the order is not relevant, it would be easier to use `set`.)
vocab = []
tf = []
for doc in tokenized_docs:
    for token in doc:
        if token not in vocab:
            vocab.append(token)
    # Count TF for each doc
    tf.append(compute_tf(doc))
tfidf = [{term: tf_doc.get(term, 0) * idf_dict.get(term, 0) for term in vocab} for tf_doc in tf]
pd.DataFrame(tfidf, columns=vocab)


,cat,dog,bird,rat,cheese,friend,human
0,0.0,0.095894,0.231049,0.000000,0.000000,0.000000,0.000000
1,0.0,0.000000,0.000000,0.346574,0.173287,0.000000,0.000000
2,0.0,0.071921,0.000000,0.000000,0.000000,0.173287,0.173287


### TF-IDF Vectorizer

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd


In [11]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)
pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())


,bird,cat,cheese,dog,friend,human,rat
0,0.720333,0.425441,0.000000,0.547832,0.000000,0.000000,0.00000
1,0.000000,0.255374,0.432385,0.000000,0.000000,0.000000,0.86477
2,0.000000,0.345205,0.000000,0.444514,0.584483,0.584483,0.00000


In [12]:
tfidf_matrix


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10 stored elements and shape (3, 7)>

## Word2Vec

In [13]:
%pip install gensim -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 34.9 MB/s eta 0:00:00


### Using pretrained model

In [14]:
import gensim.downloader as api


In [15]:
w2v_model = api.load('word2vec-ruscorpora-300')

word = "жизнь_NOUN"
if word in w2v_model.key_to_index:
    print(f"Word: '{word}':\n", w2v_model[word])
    print(f"\nVector size: {w2v_model[word].shape}")
else:
    print(f"Word '{word}' is not present in the model.")


[==================================================] 100.0% 198.8/198.8MB downloaded
Word: 'жизнь_NOUN':
 [-0.12112214  0.00870241  0.03419324  0.00739663  0.0498926   0.08822373
 -0.01143234  0.01909082  0.05713632 -0.09353109 -0.0659969  -0.0309724
 -0.030187    0.02126888 -0.00516937 -0.01718024 -0.01184916 -0.03552986
  0.00953155  0.00170796  0.06193599  0.02846407 -0.16981365  0.07623787
 -0.02958078 -0.07465961  0.01444931  0.015319   -0.00151932  0.02656073
  0.0556897   0.08897429  0.02000033 -0.01548212  0.11891276 -0.02439923
  0.07700787 -0.0455139  -0.0203667  -0.05561786  0.00300816  0.07009368
  0.02917391 -0.00023318 -0.02388291  0.07162231  0.05349505 -0.08117097
  0.14713782 -0.06134159  0.01559311 -0.02290069  0.04617056 -0.03017698
 -0.04739884  0.01343922 -0.10965177 -0.03340828  0.00238579  0.07266998
  0.04936632  0.07298899  0.03146793 -0.00565133 -0.03581524 -0.02989143
 -0.04947808 -0.05474768 -0.00945385 -0.03948738 -0.02623238 -0.05858833
  0.01024302 -0.128

In [25]:
w2v_model.key_to_index


{'весь_DET': 0,
 'человек_NOUN': 1,
 'мочь_VERB': 2,
 'год_NOUN': 3,
 'сказать_VERB': 4,
 'время_NOUN': 5,
 'говорить_VERB': 6,
 'становиться_VERB': 7,
 'знать_VERB': 8,
 'самый_DET': 9,
 'дело_NOUN': 10,
 'день_NOUN': 11,
 'жизнь_NOUN': 12,
 'рука_NOUN': 13,
 'очень_ADV': 14,
 'первый_ADJ': 15,
 'давать_VERB': 16,
 'новый_ADJ': 17,
 'слово_NOUN': 18,
 'иметь_VERB': 19,
 'большой_ADJ': 20,
 'идти_VERB': 21,
 'глаз_NOUN': 22,
 'место_NOUN': 23,
 'лицо_NOUN': 24,
 'видеть_VERB': 25,
 'хотеть_VERB': 26,
 'понимать_VERB': 27,
 'должный_ADJ': 28,
 'работа_NOUN': 29,
 'каждый_DET': 30,
 'друг_NOUN': 31,
 'голова_NOUN': 32,
 'дом_NOUN': 33,
 'оставаться_VERB': 34,
 'сторона_NOUN': 35,
 'начинать_VERB': 36,
 'думать_VERB': 37,
 'хорошо_ADV': 38,
 'жить_VERB': 39,
 'стоять_VERB': 40,
 'спрашивать_VERB': 41,
 'сделать_VERB': 42,
 'выходить_VERB': 43,
 'последний_ADJ': 44,
 'русский_ADJ': 45,
 'сила_NOUN': 46,
 'получать_VERB': 47,
 'какой-то_DET': 48,
 'хороший_ADJ': 49,
 'случай_NOUN': 50,
 'во

### Training Your Own Model

In [17]:
import re

import nltk
from gensim.models import Word2Vec
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from tqdm.auto import tqdm


In [18]:
lemmatizer = WordNetLemmatizer()
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

Dataset: https://www.google.com/url?q=https%3A%2F%2Fwww.kaggle.com%2Fdatasets%2Fteam-ai%2Fspam-text-message-classification%2F

In [19]:
df = pd.read_csv('SPAM.csv')
df


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [20]:
stop_words = set(stopwords.words('english'))

def remove_stop_words_and_lemm(text):
    word_tokens = word_tokenize(text)
    filtered_text = [lemmatizer.lemmatize(w) for w in word_tokens
                         if not w.lower() in stop_words]

    res_text = ' '.join(filtered_text)
    return res_text


In [21]:
def manual_clear(text):
    filtered_tokens = re.findall(r'[a-z]{2,}', text.lower())
    return ' '.join(filtered_tokens)


In [22]:
tqdm.pandas()

df['Message'] = df['Message'].progress_apply(remove_stop_words_and_lemm)
df['Message'] = df['Message'].progress_apply(manual_clear)
df


  0%|          | 0/5572 [00:00<?, ?it/s]

  0%|          | 0/5572 [00:00<?, ?it/s]

,Category,Message
0,ham,go jurong point crazy available bugis great wo...
1,ham,ok lar joking wif oni
2,spam,free entry wkly comp win fa cup final tkts st ...
3,ham,dun say early hor already say
4,ham,nah think go usf life around though
...,...,...
5567,spam,nd time tried contact pound prize claim easy c...
5568,ham,going esplanade fr home
5569,ham,pity mood suggestion
5570,ham,guy bitching acted like interested buying some...


In [23]:
all_words = []

df['Message'].apply(lambda x : all_words.extend(x.split()))

word_count = Counter(all_words)
word_count.most_common(20)


[('call', 627),
 ('get', 408),
 ('ur', 391),
 ('gt', 318),
 ('lt', 316),
 ('go', 315),
 ('ok', 293),
 ('free', 288),
 ('day', 281),
 ('know', 270),
 ('ll', 260),
 ('come', 254),
 ('got', 253),
 ('like', 250),
 ('good', 248),
 ('time', 246),
 ('text', 220),
 ('love', 218),
 ('want', 217),
 ('send', 200)]

Word2Vec (https://radimrehurek.com/gensim/models/word2vec.html)

**Parameters:**

- `sentences`: List of sentences as list of token lists.
- `vector_size`: Dimensionality of the embedding vectors (default: 100).
- `min_count`: Minimum word frequency to include in vocabulary.
- `window`: Maximum distance between current and predicted word.
- `workers`: Number of training threads.
- `sg`: 1 for Skip-gram, 0 for CBOW.
- `alpha`: Initial learning rate.
- `epochs`: Number of training epochs (formerly `iter`)

**Methods:**
- `build_vocab()`: Builds vocabulary from data.
- `train()`: Trains the model on sentences.
- `save()` and `load()`: Saves/loads model to/from file.
- `wv.most_similar(positive, negative)`: Finds similar words (wv is KeyedVectors).
- `wv.similarity(word1, word2)`: Computes cosine similarity between words.
- `init_sims(replace=True)`: Optimizes memory by replacing full vectors with normalized ones.

In [26]:
w2v = Word2Vec(sentences=df['Message'].apply(lambda x: x.split()), vector_size=25, min_count=1, seed=42)


In [27]:
w2v.wv['go']


array([-0.1487324 , -1.845588  , -0.39797267, -0.5477892 , -0.3390987 ,
       -0.14104205, -0.30209732,  0.519727  , -0.7741899 ,  0.19019245,
       -0.88524234,  0.6142445 , -3.0399122 ,  1.623625  ,  0.89616275,
       -0.44633138,  1.0452987 , -0.15708973, -1.7137464 ,  1.1436971 ,
       -0.2969176 ,  0.6815132 ,  0.87539905,  1.1577687 ,  0.04455895],
      dtype=float32)

## GloVe

In [28]:
import gensim.downloader as gs
import numpy as np

glove = gs.load('glove-twitter-25')


[==================================================] 100.0% 104.8/104.8MB downloaded


In [29]:
glove['cat'].shape


(25,)

Example:

Generating a vector representation of text given word embeddings. Simple realisation by counting a mean vector.

In [30]:
embedding_size = glove['cat'].shape[0]

embedding_df = pd.DataFrame([], columns=[f'emb {i}' for i in range(embedding_size)])
# iterate over all texts in dataframe
for _, row in tqdm(df.iterrows(), total=len(df)):
    text = row['Message'].split()
    # fill text matrix with vectors of words occuring in it
    emb_matrix = []
    for word in text:
        try:
            emb_word = glove[word]
        except KeyError:
            continue
            # emb_word = [0 for _ in range(embedding_size)]
        emb_matrix.append(emb_word)
    # count mean value of vectors
    avg_emb = np.mean(np.array(emb_matrix), axis=0)
    embedding_df.loc[len(embedding_df)] = avg_emb

embedding_df


  0%|          | 0/5572 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/tmp/ipython-input-2376919162.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  embedding_df.loc[len(embedding_df)] = avg_emb
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret 

,emb 0,emb 1,emb 2,emb 3,emb 4,emb 5,emb 6,emb 7,emb 8,emb 9,...,emb 15,emb 16,emb 17,emb 18,emb 19,emb 20,emb 21,emb 22,emb 23,emb 24
0,-0.282485,0.317474,0.138595,-0.348058,-0.320595,0.107828,0.517043,0.159260,0.214994,-0.169675,...,0.171143,0.350759,-0.187526,-0.463635,0.249159,-0.126669,-0.327796,-0.040991,-0.007411,0.278105
1,-0.074996,0.266892,0.377192,-0.093010,-0.574472,-0.316744,-0.152268,0.160230,-0.142340,0.508098,...,0.264926,0.432370,-0.453696,-0.614064,0.074778,0.647006,0.059942,-0.299348,0.554856,-0.174234
2,0.012178,0.739647,-0.221913,-0.508841,0.002910,-0.407856,0.367755,-0.195612,0.186368,-0.058700,...,0.050187,0.359159,-0.495891,-0.495704,-0.246067,-0.608734,-0.467419,0.232789,0.260451,-0.538218
3,0.099361,0.733045,0.247028,-0.363987,-0.398553,-0.295712,0.847007,-0.074347,-0.265625,0.556066,...,0.096179,0.773530,-0.474530,-0.524087,-0.391853,0.365488,0.558595,-0.065398,0.725585,-0.149877
4,-0.007623,0.479504,0.149703,-0.343417,-0.658401,0.211070,1.123219,-0.199515,-0.148649,0.264348,...,0.060943,0.190455,-0.837893,-0.054137,-0.162433,0.147304,0.218516,0.199540,0.072663,-0.110319
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5567,0.031726,0.728328,-0.210128,-0.190231,0.133793,-0.178479,0.623080,-0.264422,0.198138,0.080553,...,0.116530,0.291762,-0.420273,-0.482406,-0.301040,-0.267808,-0.084908,0.131469,0.103317,-0.429246
5568,-0.773277,0.724000,0.337388,-0.592416,-0.587138,0.017886,0.561440,-0.614535,0.477569,-0.042852,...,-0.136764,0.208797,-0.434453,-0.630858,-0.084565,-0.121637,-0.488671,0.229452,-0.143308,0.355256
5569,-0.035332,0.344687,-0.387425,0.424747,0.197467,0.356627,1.029587,-0.285986,0.040730,-0.135903,...,0.897503,0.744410,-0.654943,0.148100,-0.282578,0.113610,0.248770,-0.389524,0.578293,-0.680167
5570,-0.153924,0.728202,0.301370,-0.260191,0.005104,0.087682,1.049221,-0.696766,-0.450446,-0.100667,...,0.249873,0.374357,-0.558181,-0.158728,-0.398227,0.131614,-0.069433,-0.096306,0.003980,-0.516132


## FastText

In [31]:
from gensim.models import FastText


In [32]:
ft = FastText(sentences=df['Message'].apply(lambda x: x.split()), vector_size=25, window=5, min_count=1, workers=4, sg=1)


In [33]:
ft.wv['go']


array([ 0.05359192, -0.31778795,  0.8045535 , -0.00190616,  0.0262356 ,
       -0.8853365 , -0.04508992, -0.59447765, -0.9335927 , -0.08067333,
       -0.75733036, -0.4262235 ,  1.232389  , -1.0482275 , -0.11514448,
       -0.10229486,  0.40140843,  0.2121043 ,  0.12901366, -0.04674616,
       -0.24006923,  0.18314236,  0.2627432 , -0.12990817, -0.6758618 ],
      dtype=float32)

## Checking Vectors Quality

### 1. Similarity

In [34]:
from sklearn.metrics.pairwise import cosine_similarity


In [35]:
print(f"{cosine_similarity([w2v.wv['go'] + w2v.wv['fast']], [w2v.wv['run']])[0][0]: .3f}")


 0.995


In [36]:
print(f"{cosine_similarity([glove['go'] + glove['fast']], [glove['run']])[0][0]: .3f}")


 0.937


In [37]:
print(f"{cosine_similarity([ft.wv['go'] + ft.wv['fast']], [ft.wv['run']])[0][0]: .3f}")


 0.993


### 2. Visual

In [38]:
from sklearn.decomposition import PCA
import plotly.express as px


#### W2V

In [39]:
vocab = w2v.wv.index_to_key
X = w2v.wv[vocab]


In [40]:
pca = PCA(2, random_state=42)
X_transformed = pca.fit_transform(X)


In [47]:
vecs_df = pd.DataFrame(X_transformed, index=vocab, columns=['x', 'y'])

fig = px.scatter(vecs_df, x='x', y='y', hover_name=vecs_df.index, width=800, height=400)
fig.show()


#### GloVe

Unfortunately, I had to remove the output of the following cell as there are just too many vectors and building an interactive representation of them makes file too heavy so it exceeds GitHub file size limit.

In [ ]:
vocab = glove.index_to_key
X = glove[vocab]
pca = PCA(2, random_state=42)
X_transformed = pca.fit_transform(X)
vecs_df = pd.DataFrame(X_transformed, index=vocab, columns=['x', 'y'])

fig = px.scatter(vecs_df, x='x', y='y', hover_name=vecs_df.index, width=800, height=400)
fig.show()


#### FastText

In [50]:
vocab = ft.wv.index_to_key
X = ft.wv[vocab]
pca = PCA(2, random_state=42)
X_transformed = pca.fit_transform(X)
vecs_df = pd.DataFrame(X_transformed, index=vocab, columns=['x', 'y'])

fig = px.scatter(vecs_df, x='x', y='y', hover_name=vecs_df.index, width=800, height=400)
fig.show()


## Additional sources:
1. https://lena-voita.github.io/nlp_course/word_embeddings.html
2. https://habr.com/ru/articles/778048/
3. https://neerc.ifmo.ru/wiki/index.php?title=%D0%92%D0%B5%D0%BA%D1%82%D0%BE%D1%80%D0%BD%D0%BE%D0%B5_%D0%BF%D1%80%D0%B5%D0%B4%D1%81%D1%82%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5_%D1%81%D0%BB%D0%BE%D0%B2